# **Statistik Dataset CoNLL NER — Dokumen Hukum (Legal NER)**
**Menghitung distribusi B / I / O per tipe entitas dan keseluruhan.**

In [1]:
from collections import defaultdict

In [2]:
def parse_conll(filepath):
    documents = []
    current_doc = []
    skip_blank = False

    with open(filepath, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("-DOCSTART-"):
                if current_doc:
                    documents.append(current_doc)
                    current_doc = []
                skip_blank = True
                continue
            if skip_blank and line.strip() == "":
                skip_blank = False
                continue
            if line.strip() == "":
                continue
            parts = line.split()
            if len(parts) >= 2:
                current_doc.append((parts[0], parts[-1]))

    if current_doc:
        documents.append(current_doc)
    return documents


def compute_stats(documents):
    num_docs   = len(documents)
    num_tokens = sum(len(doc) for doc in documents)

    # bio_counts[label] = {"B": n, "I": n}
    bio_counts = defaultdict(lambda: {"B": 0, "I": 0})
    total_O = 0

    for doc in documents:
        for token, tag in doc:
            if tag == "O":
                total_O += 1
            elif tag.startswith("B-"):
                bio_counts[tag[2:]]["B"] += 1
            elif tag.startswith("I-"):
                bio_counts[tag[2:]]["I"] += 1

    # Urutkan berdasarkan jumlah B (jumlah entitas) descending
    sorted_labels = sorted(bio_counts.keys(), key=lambda l: -bio_counts[l]["B"])

    total_B = sum(v["B"] for v in bio_counts.values())
    total_I = sum(v["I"] for v in bio_counts.values())

    return {
        "num_docs"   : num_docs,
        "num_tokens" : num_tokens,
        "total_B"    : total_B,
        "total_I"    : total_I,
        "total_O"    : total_O,
        "bio_counts" : bio_counts,
        "sorted_labels": sorted_labels,
    }


def print_stats(split_name, stats):
    total_B = stats["total_B"]
    total_I = stats["total_I"]
    total_O = stats["total_O"]
    total   = stats["num_tokens"]

    print(f"\n{'=' * 75}")
    print(f"  SPLIT : {split_name.upper()}")
    print(f"{'=' * 75}")
    print(f"  Jumlah Dokumen   : {stats['num_docs']:,}")
    print(f"  Jumlah Token     : {total:,}")
    print(f"  Total B (awal entitas)    : {total_B:,}  ({total_B/total*100:.1f}%)")
    print(f"  Total I (dalam entitas)   : {total_I:,}  ({total_I/total*100:.1f}%)")
    print(f"  Total O (bukan entitas)   : {total_O:,}  ({total_O/total*100:.1f}%)")

    print(f"\n  {'Label':<25} {'B':>7} {'%B':>6}  {'I':>7} {'%I':>6}  {'B+I':>8}  {'Avg I/Span':>10}")
    print("  " + "-" * 72)

    for label in stats["sorted_labels"]:
        b = stats["bio_counts"][label]["B"]
        i = stats["bio_counts"][label]["I"]
        bi = b + i
        pct_b = b / total_B * 100 if total_B else 0
        pct_i = i / total_I * 100 if total_I else 0
        avg_i = i / b if b else 0
        print(f"  {label:<25} {b:>7,} {pct_b:>5.1f}%  {i:>7,} {pct_i:>5.1f}%  {bi:>8,}  {avg_i:>10.2f}")

    print("  " + "-" * 72)
    bi_total = total_B + total_I
    avg_i_total = total_I / total_B if total_B else 0
    print(f"  {'TOTAL':<25} {total_B:>7,}         {total_I:>7,}         {bi_total:>8,}  {avg_i_total:>10.2f}")
    print(f"\n  * Avg I/Span = rata-rata token I per entitas (ukuran span)")


def print_summary(all_stats):
    print(f"\n{'=' * 75}")
    print("  RINGKASAN KESELURUHAN DATASET")
    print(f"{'=' * 75}")
    print(f"  {'Split':<8} {'Dokumen':>8} {'Token':>9} {'B':>8} {'I':>8} {'O':>9}")
    print("  " + "-" * 55)
    for split, s in all_stats.items():
        print(f"  {split:<8} {s['num_docs']:>8,} {s['num_tokens']:>9,} "
              f"{s['total_B']:>8,} {s['total_I']:>8,} {s['total_O']:>9,}")
    print("  " + "-" * 55)
    print(f"  {'TOTAL':<8} "
          f"{sum(s['num_docs']    for s in all_stats.values()):>8,} "
          f"{sum(s['num_tokens']  for s in all_stats.values()):>9,} "
          f"{sum(s['total_B']     for s in all_stats.values()):>8,} "
          f"{sum(s['total_I']     for s in all_stats.values()):>8,} "
          f"{sum(s['total_O']     for s in all_stats.values()):>9,}")


In [3]:
# ── Main ──────────────────────────────────────────────────────────────────────

FILES = {
    "train" : "/content/drive/MyDrive/data_conll_100_v6.1/train.conll",
    "dev"   : "/content/drive/MyDrive/data_conll_100_v6.1/dev.conll",
    "test"  : "/content/drive/MyDrive/data_conll_100_v6.1/test.conll",
}

all_stats = {}
for split, path in FILES.items():
    documents = parse_conll(path)
    stats     = compute_stats(documents)
    all_stats[split] = stats
    print_stats(split, stats)

print_summary(all_stats)


  SPLIT : TRAIN
  Jumlah Dokumen   : 1
  Jumlah Token     : 69,004
  Total B (awal entitas)    : 8,774  (12.7%)
  Total I (dalam entitas)   : 23,522  (34.1%)
  Total O (bukan entitas)   : 36,708  (53.2%)

  Label                           B     %B        I     %I       B+I  Avg I/Span
  ------------------------------------------------------------------------
  LAW_NAME                    1,505  17.2%    8,120  34.5%     9,625        5.40
  JUDGE                       1,202  13.7%    1,474   6.3%     2,676        1.23
  PARTY_REF                   1,192  13.6%      862   3.7%     2,054        0.72
  ORG                         1,099  12.5%    1,188   5.1%     2,287        1.08
  LAW_CONS_ART                  767   8.7%    2,461  10.5%     3,228        3.21
  LEGAL_DOC                     690   7.9%    2,533  10.8%     3,223        3.67
  LEGAL_DOC_NUM                 535   6.1%      469   2.0%     1,004        0.88
  DATE                          437   5.0%      893   3.8%     1,330   